1. Setup
Installs PySpark and starts the Spark session.

In [ ]:

%pip install pyspark


In [ ]:

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("EcommerceSalesETL")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark session started successfully!")


Spark version: 4.0.4
Spark session started successfully!


2. Create Raw Data
Generates sample e-commerce data with intentional mistakes (like missing values, negatives, and duplicates) and saves it as a CSV file.

In [ ]:

import csv
import random
from datetime import datetime, timedelta

random.seed(42)

products = {
    "Laptop": 45000,
    "Phone": 25000,
    "Headphones": 2000,
    "Keyboard": 1500,
    "Monitor": 12000
}

statuses = ["Delivered", "Shipped", "Cancelled", "Returned"]

base_rows = []
start_date = datetime(2026, 1, 1)

for i in range(9950):
    product = random.choice(list(products.keys()))
    quantity = random.randint(1, 5)
    unit_price = round(
        products[product] * random.uniform(0.85, 1.15), 2
    )

    # Intentionally introduce data-quality issues
    if i % 149 == 0:
        unit_price = None
    elif i % 307 == 0:
        unit_price = -abs(unit_price)

    if i % 211 == 0:
        quantity = 0

    customer_id = None if i % 101 == 0 else f"C{1000 + i}"

    order_date = (
        start_date + timedelta(days=random.randint(0, 89))
    ).strftime("%Y-%m-%d")

    base_rows.append({
        "order_id": f"ORD{10000 + i}",
        "order_date": order_date,
        "customer_id": customer_id,
        "product": product,
        "quantity": quantity,
        "unit_price": unit_price,
        "status": random.choice(statuses)
    })

# Add 50 duplicate records
raw_rows = base_rows + [dict(row) for row in base_rows[:50]]

random.shuffle(raw_rows)

with open("/content/orders_raw.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=[
        "order_id",
        "order_date",
        "customer_id",
        "product",
        "quantity",
        "unit_price",
        "status"
    ])
    writer.writeheader()
    writer.writerows(raw_rows)

print("Dataset created successfully!")
print("Total rows:", len(raw_rows))
print("CSV saved at /content/orders_raw.csv")


Dataset created successfully!
Total rows: 10000
CSV saved at /content/orders_raw.csv


In [ ]:

with open("/content/orders_raw.csv", "r") as file:
    for i in range(6):
        print(file.readline().strip())


order_id,order_date,customer_id,product,quantity,unit_price,status
ORD19623,2026-03-17,C10623,Keyboard,2,1485.56,Returned
ORD17771,2026-01-08,C8771,Phone,4,23551.64,Shipped
ORD11353,2026-01-30,C2353,Headphones,2,1737.77,Cancelled
ORD13575,2026-03-13,C4575,Monitor,3,13642.53,Cancelled
ORD14315,2026-02-13,C5315,Keyboard,4,1398.42,Returned


3. Bronze Layer
Reads the raw CSV file and stores it as a Parquet file without any changes.

In [ ]:

from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)

order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("status", StringType(), True)
])

bronze_df = (
    spark.read
    .schema(order_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .csv("/content/orders_raw.csv")
)

bronze_df.printSchema()
bronze_df.show(5, truncate=False)

print("Bronze row count:", bronze_df.count())


root
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)

+--------+----------+-----------+----------+--------+----------+---------+
|order_id|order_date|customer_id|product   |quantity|unit_price|status   |
+--------+----------+-----------+----------+--------+----------+---------+
|ORD19623|2026-03-17|C10623     |Keyboard  |2       |1485.56   |Returned |
|ORD17771|2026-01-08|C8771      |Phone     |4       |23551.64  |Shipped  |
|ORD11353|2026-01-30|C2353      |Headphones|2       |1737.77   |Cancelled|
|ORD13575|2026-03-13|C4575      |Monitor   |3       |13642.53  |Cancelled|
|ORD14315|2026-02-13|C5315      |Keyboard  |4       |1398.42   |Returned |
+--------+----------+-----------+----------+--------+----------+---------+
only showing top 5 rows
Bronze row

In [ ]:

bronze_path = "/content/bronze_orders"

bronze_df.write.mode("overwrite").parquet(bronze_path)

print("Bronze data saved successfully!")


Bronze data saved successfully!


In [ ]:

from pyspark.sql.functions import (
    col, trim, to_date, initcap,
    count, sum as spark_sum, round as spark_round,
    row_number
)

from pyspark.sql.window import Window


In [ ]:

print("Total raw rows:", bronze_df.count())
print("Unique order IDs:", bronze_df.select("order_id").distinct().count())

bronze_df.select(
    spark_sum(col("customer_id").isNull().cast("int")).alias("missing_customer_ids"),
    spark_sum(col("unit_price").isNull().cast("int")).alias("missing_prices"),
    spark_sum((col("quantity") <= 0).cast("int")).alias("invalid_quantities"),
    spark_sum((col("unit_price") <= 0).cast("int")).alias("invalid_prices")
).show()


Total raw rows: 10000
Unique order IDs: 9950
+--------------------+--------------+------------------+--------------+
|missing_customer_ids|missing_prices|invalid_quantities|invalid_prices|
+--------------------+--------------+------------------+--------------+
|                 100|            68|                49|            32|
+--------------------+--------------+------------------+--------------+



In [ ]:

dedup_df = bronze_df.dropDuplicates(["order_id"])

print("Before deduplication:", bronze_df.count())
print("After deduplication:", dedup_df.count())
print(
    "Duplicate rows removed:",
    bronze_df.count() - dedup_df.count()
)


Before deduplication: 10000
After deduplication: 9950
Duplicate rows removed: 50


4. Silver Layer
Cleans the data, removes duplicates, fixes text format, separates good data from bad data, and checks it with code rules.

In [ ]:

prepared_df = (
    dedup_df
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("product", initcap(trim(col("product"))))
    .withColumn("status", initcap(trim(col("status"))))
    .withColumn(
        "order_date",
        to_date(trim(col("order_date")), "yyyy-MM-dd")
    )
)

valid_condition = (
    col("order_id").isNotNull()
    & (col("order_id") != "")
    & col("order_date").isNotNull()
    & col("customer_id").isNotNull()
    & (col("customer_id") != "")
    & col("product").isNotNull()
    & (col("product") != "")
    & col("quantity").isNotNull()
    & (col("quantity") > 0)
    & col("unit_price").isNotNull()
    & (col("unit_price") > 0)
    & col("status").isin(
        "Delivered", "Shipped", "Cancelled", "Returned"
    )
)

silver_df = prepared_df.filter(valid_condition)
rejected_df = prepared_df.filter(~valid_condition)

print("Deduplicated rows:", dedup_df.count())
print("Valid Silver rows:", silver_df.count())
print("Rejected rows:", rejected_df.count())


Deduplicated rows: 9950
Valid Silver rows: 9706
Rejected rows: 244


In [ ]:

silver_df.printSchema()
silver_df.show(10, truncate=False)


root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- status: string (nullable = true)

+--------+----------+-----------+-------+--------+----------+---------+
|order_id|order_date|customer_id|product|quantity|unit_price|status   |
+--------+----------+-----------+-------+--------+----------+---------+
|ORD10001|2026-03-11|C1001      |Phone  |1       |26325.25  |Delivered|
|ORD10002|2026-01-12|C1002      |Monitor|4       |10314.42  |Shipped  |
|ORD10003|2026-03-13|C1003      |Phone  |5       |25765.14  |Shipped  |
|ORD10004|2026-03-17|C1004      |Monitor|4       |10993.59  |Cancelled|
|ORD10005|2026-02-13|C1005      |Laptop |2       |47674.88  |Cancelled|
|ORD10006|2026-02-13|C1006      |Phone  |2       |28429.1   |Delivered|
|ORD10007|2026-02-14|C1007      |Laptop |4       |39555.67  |Ca

In [ ]:

assert silver_df.count() > 0

assert silver_df.filter(
    col("order_id").isNull()
    | (col("order_id") == "")
).count() == 0

assert silver_df.groupBy("order_id").count().filter(
    col("count") > 1
).count() == 0

assert silver_df.filter(
    col("customer_id").isNull()
    | col("product").isNull()
    | col("order_date").isNull()
    | col("quantity").isNull()
    | (col("quantity") <= 0)
    | col("unit_price").isNull()
    | (col("unit_price") <= 0)
).count() == 0

print("All Silver validation checks passed!")


All Silver validation checks passed!


In [ ]:

silver_df.write.mode("overwrite").parquet(
    "/content/silver_orders"
)

rejected_df.write.mode("overwrite").parquet(
    "/content/rejected_orders"
)

print("Silver and rejected data saved!")


Silver and rejected data saved!


In [ ]:

silver_df.createOrReplaceTempView("silver_orders")

print("SQL view created: silver_orders")


SQL view created: silver_orders


5. Gold Layer
Uses SQL to create useful business reports (like daily sales and product totals) and saves them

In [ ]:

daily_sales_df = spark.sql("""
    SELECT
        order_date,
        COUNT(*) AS delivered_orders,
        ROUND(SUM(quantity * unit_price), 2) AS daily_revenue
    FROM silver_orders
    WHERE status = 'Delivered'
    GROUP BY order_date
    ORDER BY order_date
""")

daily_sales_df.show(10, truncate=False)


+----------+----------------+-------------+
|order_date|delivered_orders|daily_revenue|
+----------+----------------+-------------+
|2026-01-01|34              |2203222.75   |
|2026-01-02|27              |1282375.22   |
|2026-01-03|24              |1374843.4    |
|2026-01-04|24              |1153007.57   |
|2026-01-05|30              |1932993.89   |
|2026-01-06|25              |1143245.91   |
|2026-01-07|29              |1150369.05   |
|2026-01-08|20              |1280036.44   |
|2026-01-09|21              |1285265.11   |
|2026-01-10|25              |1047141.71   |
+----------+----------------+-------------+
only showing top 10 rows


In [ ]:

product_sales_df = spark.sql("""
    SELECT
        product,
        SUM(quantity) AS units_sold,
        ROUND(SUM(quantity * unit_price), 2) AS total_revenue
    FROM silver_orders
    WHERE status = 'Delivered'
    GROUP BY product
    ORDER BY total_revenue DESC
""")

product_sales_df.show(truncate=False)


+----------+----------+-------------+
|product   |units_sold|total_revenue|
+----------+----------+-------------+
|Laptop    |1485      |6.684536223E7|
|Phone     |1459      |3.665216997E7|
|Monitor   |1394      |1.667772634E7|
|Headphones|1268      |2524294.36   |
|Keyboard  |1523      |2285525.6    |
+----------+----------+-------------+



In [ ]:

status_report_df = spark.sql("""
    SELECT
        status,
        COUNT(*) AS order_count
    FROM silver_orders
    GROUP BY status
    ORDER BY order_count DESC
""")

status_report_df.show()


+---------+-----------+
|   status|order_count|
+---------+-----------+
| Returned|       2497|
|  Shipped|       2410|
|Cancelled|       2405|
|Delivered|       2394|
+---------+-----------+



In [ ]:

daily_sales_df.write.mode("overwrite").parquet(
    "/content/gold_daily_sales"
)

product_sales_df.write.mode("overwrite").parquet(
    "/content/gold_product_sales"
)

status_report_df.write.mode("overwrite").parquet(
    "/content/gold_order_status"
)

print("Gold analytics tables saved successfully!")


Gold analytics tables saved successfully!


In [ ]:

print("BRONZE:", bronze_df.count())
print("SILVER:", silver_df.count())
print("REJECTED:", rejected_df.count())

print("GOLD daily sales rows:", daily_sales_df.count())
print("GOLD product rows:", product_sales_df.count())
print("GOLD status rows:", status_report_df.count())

assert bronze_df.count() == 10000

assert bronze_df.count() == (
    bronze_df.dropDuplicates(["order_id"]).count() + 50
)

assert bronze_df.dropDuplicates(["order_id"]).count() == (
    silver_df.count() + rejected_df.count()
)

assert product_sales_df.count() > 0
assert status_report_df.count() > 0

print("Pipeline checks completed successfully!")


BRONZE: 10000
SILVER: 9706
REJECTED: 244
GOLD daily sales rows: 90
GOLD product rows: 5
GOLD status rows: 4
Pipeline checks completed successfully!


6. Export
Converts the final report into a single CSV file and downloads it to your computer.

In [ ]:

import os

for folder in [
    "/content/bronze_orders",
    "/content/silver_orders",
    "/content/rejected_orders",
    "/content/gold_daily_sales",
    "/content/gold_product_sales",
    "/content/gold_order_status"
]:
    print(folder, "exists:", os.path.exists(folder))


/content/bronze_orders exists: True
/content/silver_orders exists: True
/content/rejected_orders exists: True
/content/gold_daily_sales exists: True
/content/gold_product_sales exists: True
/content/gold_order_status exists: True


In [ ]:

import glob
import shutil
from google.colab import files

daily_sales_df.coalesce(1).write.mode("overwrite") \
    .option("header", True) \
    .csv("/content/daily_sales_export")

part_files = glob.glob(
    "/content/daily_sales_export/part-*.csv"
)

assert len(part_files) == 1, "Expected one CSV part file"

shutil.copy(part_files[0], "/content/daily_sales_report.csv")

print("CSV export ready!")
files.download("/content/daily_sales_report.csv")


CSV export ready!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>